# Linear Regression and Regularization

In [ ]:
import importlib
from IPython import get_ipython


dependencies = ["numpy", "pandas", "matplotlib", "seaborn", "scikit-learn", "hashlib", "scikit-learn"]

for pkg in dependencies:
    try:
        importlib.import_module(pkg)
        print(f"{pkg} already installed")
    except ImportError:
        print(f"{pkg} not found. Installing...")
        get_ipython().run_line_magic("pip", f"install {pkg}")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import hashlib

print("All dependencies successfully imported")

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder,StandardScaler

In [ ]:
username = "gracy.garg"
seed = int ( hashlib . sha256 ( username . encode () ) . hexdigest () , 16)
seed = seed % (2**32)
# print(f"Seed for {username}: {seed}")

# > Include username in plots

In [ ]:
def add_username_to_plot(username="gracy.garg"):
    """
    Adds the username to the top-right corner of the current plot.
    """
    plt.text(
        0.95, 0.95, username,
        ha='right', va='top',
        transform=plt.gca().transAxes,
        fontsize=10, color='gray', alpha=0.7
    )

# Q1  Linear Regression with Regularization 

In [ ]:
student_df = pd.read_csv("student_dataset.csv")
X = student_df.drop(columns=["gpa"])
y = student_df["gpa"]

# Ordinal encode 'major'
ordinal_cols = ["major","program"]
ordinal_encoder = OrdinalEncoder()
X[ordinal_cols] = ordinal_encoder.fit_transform(X[ordinal_cols])



# One-hot encode 'gender'
X = pd.get_dummies(X, columns=["gender"], drop_first=True)
# Step 2: One-hot encode categorical features

# Step 3: Train/val/test split (60/20/20 for example)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

In [ ]:

def run_poly_regression(X_train, y_train,
                        X_val, y_val,
                        X_test, y_test,
                        degree=1,
                        regularizer=None,
                        reg_strength=0.0):
    """
    Fit a polynomial regression model with optional regularization.

    Parameters:
    -----------
    degree ( int ) : Degree of the polynomial to fit
    regularizer ( str or None ) : l1, l2, or None
    reg_strength ( float ) : Regularization coefficient ( alpha )

    Returns:
    --------
    dict with train, val, and test MSEs, and learned coefficients
    """

    # Polynomial expansion
    
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_train_poly = poly.fit_transform(X_train)
    X_val_poly   = poly.transform(X_val)
    X_test_poly  = poly.transform(X_test)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_poly)
    X_val_scaled   = scaler.transform(X_val_poly)
    X_test_scaled  = scaler.transform(X_test_poly)

    # Choose model
    if regularizer is None:
        model = LinearRegression()
    elif regularizer == 'l1':
        model = Lasso(alpha=reg_strength, max_iter=5000, random_state=42)
    elif regularizer == 'l2':
        model = Ridge(alpha=reg_strength, max_iter=5000, random_state=42)
    else:
        raise ValueError("regularizer must be None, 'l1', or 'l2'")

    # Fit model
    model.fit(X_train_scaled, y_train)

    # Predictions
    y_train_pred = model.predict(X_train_scaled)
    y_val_pred   = model.predict(X_val_scaled)
    y_test_pred  = model.predict(X_test_scaled)

    # Compute MSE
    results = {
        "train_mse": mean_squared_error(y_train, y_train_pred),
        "val_mse": mean_squared_error(y_val, y_val_pred),
        "test_mse": mean_squared_error(y_test, y_test_pred),
        "coefficients": model.coef_,
        "features": poly.get_feature_names_out(input_features=X_train.columns)
    }
    return results


In [ ]:
def sweep_degrees(X_train, y_train, X_val, y_val, X_test, y_test,
                  regularizer=None, reg_strength=0.0,
                  degrees=range(1,7)):
    """
    Train polynomial regression across multiple degrees and record MSEs.
    """
    results = []
    for d in degrees:
        res = run_poly_regression(X_train, y_train,
                                  X_val, y_val,
                                  X_test, y_test,
                                  degree=d,
                                  regularizer=regularizer,
                                  reg_strength=reg_strength)
        results.append({
            "degree": d,
            "train_mse": res["train_mse"],
            "val_mse": res["val_mse"],
            "test_mse": res["test_mse"]
        })
    return pd.DataFrame(results)

def sweep_regularization(X_train, y_train, X_val, y_val, X_test, y_test,
                         degree, regularizer, strengths=np.logspace(-4,2,10)):
    """
    Sweep over reg_strength values (alphas) for a fixed degree and regularizer.
    """
    results = []
    for alpha in strengths:
        res = run_poly_regression(X_train, y_train,
                                  X_val, y_val,
                                  X_test, y_test,
                                  degree=degree,
                                  regularizer=regularizer,
                                  reg_strength=alpha)
        results.append({
            "alpha": alpha,
            "train_mse": res["train_mse"],
            "val_mse": res["val_mse"],
            "test_mse": res["test_mse"]
        })
    return pd.DataFrame(results)

In [ ]:
regularizers = [None, "l1", "l2"]
degree_results_dict = {}

for reg in regularizers:
    print(f"\n=== Regularizer: {reg} ===")
    deg_results = sweep_degrees(X_train, y_train, X_val, y_val, X_test, y_test,
                                regularizer=reg, reg_strength=100)
    degree_results_dict[reg] = deg_results

    # Plot degree vs MSE
    plt.figure(figsize=(6,4))
    plt.plot(deg_results["degree"], deg_results["train_mse"], marker='o', label="Train MSE")
    plt.plot(deg_results["degree"], deg_results["val_mse"], marker='o', label="Validation MSE")
    plt.xlabel("Polynomial Degree")
    plt.ylabel("MSE")
    plt.title(f"Polynomial Degree vs MSE ({'No regularization' if reg is None else reg.upper()})")
    add_username_to_plot(username)
    plt.legend()
    plt.show()

# Observations from Polynomial Degree vs MSE

L1 Regularization (Lasso):

Validation MSE decreases until degree 2, then remains nearly flat for higher degrees.

This indicates that increasing the degree beyond 2 does not improve predictive performance because L1 penalization forces less important coefficients to zero, effectively limiting model complexity.

L2 Regularization (Ridge) & No Regularization:

Validation MSE is almost flat across all degrees, showing that adding polynomial terms beyond degree 1–2 does not significantly reduce error.

This may indicate that the dataset’s relationship with GPA is relatively simple, and higher-degree polynomials do not provide additional predictive power.

Interpretation:

L1’s flat curve after degree 2 suggests it successfully controls model complexity and removes unimportant polynomial features.

L2 and unregularized models do not eliminate features, so the model uses all polynomial terms, but extra terms do not improve validation performance.

Overall, degree 2 with L1 regularization seems sufficient to capture the main signal in the data while keeping the model sparse and interpretable.

In [ ]:
# (b) Choose best degree from val curve 
deg_results_l1 = sweep_degrees(
    X_train, y_train, X_val, y_val, X_test, y_test,
    regularizer="l1", reg_strength=0.1
)
best_degree = int(deg_results_l1.loc[deg_results_l1["val_mse"].idxmin(), "degree"])
print(f"Best degree based on L1 validation: {best_degree}")

deg_results_l2 = sweep_degrees(
    X_train, y_train, X_val, y_val, X_test, y_test,
    regularizer="l2", reg_strength=100
)
best_degree = int(deg_results_l2.loc[deg_results_l2["val_mse"].idxmin(), "degree"])
print(f"Best degree based on L2 validation: {best_degree}")


In [ ]:
degrees = range(1, 7)
alphas = np.logspace(-4, 2, 10)  # candidate regularization strengths
results = []

# Collect results
for deg in degrees:
    for reg in ["l1", "l2"]:
        alpha_results = sweep_regularization(
            X_train, y_train, X_val, y_val, X_test, y_test,
            degree=deg, regularizer=reg, strengths=alphas
        )
        best_alpha = alpha_results.loc[alpha_results["val_mse"].idxmin(), "alpha"]
        best_val_mse = alpha_results["val_mse"].min()
        results.append({
            "degree": deg,
            "regularizer": reg,
            "best_alpha": best_alpha,
            "val_mse": best_val_mse
        })

results_df = pd.DataFrame(results)

# -----------------------
# Plot validation MSE vs degree
# -----------------------
plt.figure(figsize=(8, 6))
for reg in ["l1", "l2"]:
    subset = results_df[results_df["regularizer"] == reg]
    plt.plot(subset["degree"], subset["val_mse"], marker="o", label=f"{reg.upper()}")

plt.xlabel("Polynomial Degree")
plt.ylabel("Best Validation MSE")
plt.title("Best Validation MSE vs Polynomial Degree")
plt.xticks(degrees)
plt.grid(True)
add_username_to_plot(username)
plt.legend()
plt.show()


The validation MSE decreases sharply from degree 1 to 2, then remains almost constant for higher degrees, indicating that increasing polynomial complexity beyond degree 2 provides little improvement. L1 and L2 regularization select different optimal alphas, but the best validation performance is nearly identical across setups.

In [ ]:
# (c) Sweep reg strengths for that degree
alpha_results = sweep_regularization(
    X_train, y_train, X_val, y_val, X_test, y_test,
    degree=best_degree,
    regularizer="l1",
    strengths=np.logspace(-4, 2, 10)
)

# Plot
plt.semilogx(alpha_results["alpha"], alpha_results["val_mse"], marker="o", label="Validation MSE")
plt.semilogx(alpha_results["alpha"], alpha_results["train_mse"], marker="o", label="Train MSE")
plt.xlabel("Regularization Strength (log scale)")
plt.ylabel("MSE")
plt.title(f"Val/Train MSE vs Regularization Strength (Degree={best_degree}, L2)")
plt.legend()
add_username_to_plot("gracy.garg")
plt.show()

alpha_results = sweep_regularization(
    X_train, y_train, X_val, y_val, X_test, y_test,
    degree=best_degree,
    regularizer="l2",
    strengths=np.logspace(-4, 2, 10)
)

# Plot
plt.semilogx(alpha_results["alpha"], alpha_results["val_mse"], marker="o", label="Validation MSE")
plt.semilogx(alpha_results["alpha"], alpha_results["train_mse"], marker="o", label="Train MSE")
plt.xlabel("Regularization Strength (log scale)")
plt.ylabel("MSE")
plt.title(f"Val/Train MSE vs Regularization Strength (Degree={best_degree}, L2)")
plt.legend()
add_username_to_plot("gracy.garg")
plt.show()


### Performance improvement from regularization
| Degree | L1 best val MSE | L2 best val MSE |
| ------ | --------------- | --------------- |
| 1      | 0.8198          | 0.81997         |
| 2      | 0.80475         | 0.80468         |
| 3–6    | ~0.8047–0.8052  | ~0.8047–0.8052  |

Observation:

Validation MSE decreases sharply from degree 1 → 2, indicating that increasing polynomial complexity improves fit.

Beyond degree 2, the MSE stabilizes (
Δval MSE<10−3
Δval MSE<10
−3
), suggesting higher degrees add negligible benefit.

Regularization (L1 or L2) slightly adjusts the optimal alpha (
α
α) to reduce overfitting, but the overall gain is minimal, showing that the dataset is not highly overparameterized.

Best setup for test performance:

Degree 2, either L1 (
α=10−4
α=10
−4
) or L2 (
α=10−4
α=10
−4
).

Test MSE for both is almost identical (~0.8047).

### Feature importance for L1 regularization Observation:
Only a subset of polynomial features contributes to predicting GPA.
These include main effects and a few second-degree interactions between key predictors like major, program, or gender.

### Feature importance for L2 regularization Observation:
L2 includes all features but assigns small coefficients to less important predictors.
The top contributors overlap with the L1-selected features, but L2 shows smoother weight distribution and keeps weak features instead of zeroing them.

| Regularizer | Sparsity | Feature selection                     | Weight distribution                        |
| ----------- | -------- | ------------------------------------- | ------------------------------------------ |
| L1          | Sparse   | Only most important features retained | Sharp, some zeros                          |
| L2          | Dense    | All features retained                 | Smooth, small weights for minor predictors |


### Conclusion:

Regularization helps slightly in preventing overfitting, especially for higher-degree polynomial features.
L1 is preferable if interpretability is important, as it highlights the key predictors for GPA
L2 is preferable if we want all predictors to contribute, but with controlled magnitudes.